# Objectif du notebook 

Date de création : 13/01/2026

Estimation des temps d'arrivée des signaux sur la voie hydro des OBS ("H"). 

In [1]:
import os
import sys
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import gc 

In [2]:
project_root = os.path.abspath(
    os.path.join(os.path.dirname(os.getcwd()), "..", "..", "..")
)
root_groix_data = os.path.join(project_root, "data", "fiberscope_groix_oct_2025")
root_groix_wav = os.path.join(root_groix_data, "wav")
root_groix_metadata = os.path.join(root_groix_data, "metadata")

root_folder = os.path.join(project_root, "real_data_analysis", "fiberscope_groix")
data_folder = os.path.join(root_folder, "data")
img_folder = os.path.join(root_folder, "img")

In [3]:
sys.path.append(project_root)
from publication.publication_figure import PubFigure, color
from real_data_analysis.fiberscope_groix.src.data_processing.arrivals_utils import *

# Chargement des données utiles 

In [4]:
ds_gps = xr.open_dataset(os.path.join(data_folder, "gps.nc"))

# Exemple avec un unique signal

## Chargement des informations relatives aux émissions 

In [5]:
fpath = os.path.join(data_folder, "processed_emissions.nc")
ds_emis = xr.open_dataset(fpath)
df_emis = ds_emis.to_dataframe()

### Remarque : calcul de la position de la source 
Le dataframe df contient la position de l'antenne GPS estimée à l'instant de l'émission. En pratique, la source n'est pas colocalisée avec l'antenne et il faut théoriquement prendre en compte ce bras de levier.

Au moins dans le cas des émissions en statique on peut considérer, au regard de l'incertitude sur le positionnement de l'antenne GPS à l'instant émission (incertitude GPS standard + interpolation linéaire entre deux points GPS), que la source est colocalisée avec l'antenne GPS. 

Dans le cas dynamique la longueur filée est importante $\approx$ 15/20 m. Dans ce cas, il peut être plus difficile de négliger le bras de levier entre l'antenne GPS et la source immergée. La longueur filée est connue ainsi que l'immersion de la source (capteur de pression sur la source), on peut ainsi calculer la distance (à la surface) de l'antenne GPS à la source (dans l'axe du navire). Néanmoins, afin de transformer ce bras de levier dans le repère du navire en un offset sur la position dans le repère ENU il est nécessaire de connaitre le cap du navire dans le repère ENU. 

Pour ce faire, on peut estimer le cap du navire à partir de l'estimation du vecteur vitesse du navire à l'instant d'émission : 

$$ V_{gps}^{(ENU)} = [V_e, V_n]^T$$

$$V_e = \frac{E_{gps}(t_{n+1}) - E_{gps}(t_n)}{\Delta t}$$
et 
$$V_n = \frac{N_{gps}(t_{n+1}) - N_{gps}(t_n)}{\Delta t}$$

où $t_n$ et $t_{n+1}$ sont les instants précèdent et successif à l'instant d'émission dans la série temporelle des positions GPS. 

Le cap du navire dans le repère ENU est alors donné par : 

$$\alpha = \arctan{\frac{V_e}{V_n}}$$

La position dans le repère du navire est (hypothèse source dans l'axe du navire):

$$X_s^{(Navire)} = [0, -bdl]^T$$

où $bdl$ est la distance, selon l'axe $y_{navire}$, de l'origine du repère du navire (position de l'antenne GPS) au projeté orthogonal de la position de la source sur la surface. 

Finalement :

$$X_s^{(ENU)} = X_{GPS}^{(ENU)} + R_{\text{Nav to ENU}} X_s^{(Navire)}$$

avec 
$$ R_{\text{Nav to ENU}} = \begin{bmatrix} \cos{\alpha} & \sin{\alpha} \\ -\sin{\alpha} & \cos{\alpha} \end{bmatrix}$$

Proche des OBS la correction pourrait avoir un impacte significatif. 

In [6]:
pfig = PubFigure(
    label_fontsize=18, legend_fontsize=10, ticks_fontsize=16, title_fontsize=20
)

## Estimation du décalage des deux bases de temps UTC 

Le temps UTC de l'hydro source, utilisé pour pointer les temps d'émission, n'est pas parfaitement synchronisé avec le temps UTC GPS (celui des OBS). L'objectif est d'exploiter les émissions au-dessus de chacun des OBSs afin d'estimer ce décalage. 

Cette étape préalable est nécessaire pour la suite de l'estimation des temps d'arrivée. En effet certaine des émissions ne sont pas détectées, dans ce cas, il faut associer les arrivées éparses détectées aux émissions correspondantes. L'alignement des deux bases de temps est nécessaire à cette étape permettant de renforcer la robustesse de la méthode. 

### Détails

* Les temps d'arrivées théoriques sont donnés en temps UTC de l'hydrophone : $t_{th_{arr}}^{(Hydro)}$
* Les temps d'arrivées mesurés sont donnés en temps UTC de l'OBS : $t_{arr}^{(OBS)}$

Le shift entre les deux bases de temps est donné (aux erreurs de mesures et de modélisation près) par : 
$$ \tau_{Hydro} = t_{th_{arr}}^{(Hydro)} - t_{arr}^{(OBS)}$$

Ici le shift est évalué sur la différence de temps de propagation : 

* Le temps de propagation théorique est donné par : 
$$\tau_{th} = t_{th_{arr}}^{(Hydro)} - t_{emission}^{(Hydro)}$$ 
* Le temps de propagation mesuré est donné par :
$$\tau_{mes} = t_{arr}^{(OBS)} - t_{emission}^{(Hydro)}$$

et on a donc : 

$$\tau_{Hydro} = \tau_{th} - \tau_{mes} $$

### Sélection d'une partie des émissions

In [7]:
# subset_params = {
#     "Signal": "chirp",  # "chirp" or "sinus"
#     "Source": "fixed",  # "fixed" or "trailed"
#     "Nrepeat": 10,  # number of repeats
#     "Vc carte (V)": None,  # Source amplitude (V)
#     "Emission datetime": None,  # Specific day to select (datetime object) e.g datetime(2025, 10, 16)
# }

subset_params = {
    "Signal": "chirp",  # "chirp" or "sinus"
    "Source": None,  # "fixed" or "trailed"
    "Nrepeat": None,  # number of repeats
    "Vc carte (V)": None,  # Source amplitude (V)
    "Emission datetime": None,  # Specific day to select (datetime object) e.g datetime(2025, 10, 16)
}

df_sel = select_dataframe_subset(df_emis, subset_params)

In [8]:
df_sel

,Emission datetime,Sequence_id,Point,Source,Longueur filée (m),Hydro distance (m),Signal,Frequency min (Hz),Frequency max (Hz),Duration (s),...,Emission longitude AIS,Emission E AIS,Emission N AIS,Emission U AIS,Emission interpolated E GPS,Emission interpolated N GPS,Emission interpolated U GPS,Emission interpolated E AIS,Emission interpolated N AIS,Emission interpolated U AIS
0,2025-10-14 09:03:04.663609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,-3.427327,2892.678727,-633.368176,30.474371,2873.381367,-640.188097,30.482395,2895.350095,-634.274263,30.473070
1,2025-10-14 09:03:14.663609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,-3.427251,2898.406840,-635.311063,30.471582,2878.702910,-642.656231,30.502603,2903.662341,-637.758461,30.491800
2,2025-10-14 09:03:24.663609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,-3.427101,2909.676012,-640.558926,30.514935,2884.024454,-645.124365,30.526102,2912.260135,-642.100236,30.513601
3,2025-10-14 09:03:34.663609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,-3.427027,2915.217049,-643.863898,30.512076,2889.345986,-647.592497,30.523448,2917.801173,-645.405207,30.510740
4,2025-10-14 09:03:44.663609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,-3.426954,2920.758087,-647.168869,30.509211,2897.424645,-650.577764,30.496632,2926.839583,-648.825726,30.483404
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1982,2025-10-16 14:47:04.083187500,168,1S,fixed,8,1.35,chirp,80.0,1000.0,1.0,...,-3.476392,-792.555567,-806.813932,30.722366,-779.867639,-809.583654,30.723575,-794.478684,-807.744351,30.722009
1983,2025-10-16 14:47:06.083187500,168,1S,fixed,8,1.35,chirp,80.0,1000.0,1.0,...,-3.476454,-797.265410,-809.092589,30.721491,-781.069500,-810.070444,30.723366,-795.420653,-808.200082,30.721834
1984,2025-10-16 14:47:08.083187500,168,1S,fixed,8,1.35,chirp,80.0,1000.0,1.0,...,-3.476454,-797.265410,-809.092589,30.721491,-782.271361,-810.557233,30.723157,-796.362622,-808.655813,30.721659
1985,2025-10-16 14:47:10.083187500,168,1S,fixed,8,1.35,chirp,80.0,1000.0,1.0,...,-3.476454,-797.265410,-809.092589,30.721491,-783.473222,-811.044023,30.722949,-797.340209,-809.128651,30.721477


In [9]:
plot = False
savefig = False
verbose = False
plot_zoom = False

# Define image folder for preprocessing plots
img_process_folder = os.path.join(
    img_folder, "reception", "arrivals_detection", "processed_emissions"
)

# Convention Gen_Axes_D_V4 (Cf ELOBSBin2Wav.py)
channels_order = {
    "Z": 0,
    "X": 1,
    "Y": 2,
    "H": 3,
}
used_channel = "H"

# TODO : check to remove this or to moove it earlier
t_hydro_source_offset = 27  # seconds

# Window parameters
pre_reception_time = 5.0  # seconds before reception to include in the window
post_reception_time = 10.0  # seconds after reception to include in the window

# Correct window for hydrophone to source offset
pre_reception_time -= t_hydro_source_offset
post_reception_time += t_hydro_source_offset

sel_sequence_id = df_sel["Sequence_id"].unique()

print(sel_sequence_id)
sel_sequence_id = [91  95]
# sel_sequence_id = [str(seq_id) for seq_id in sel_sequence_id]

[  1   2  12  13  14  15  16  24  25  26  27  28  29  35  39  40  41  42
  43  44  46  47  51  55  59  60  61  62  63  64  69  70  71  72  76  80
  81  82  83  84  85  90  91  95  99 100 101 102 103 104 106 110 116 117
 118 119 120 121 123 127 131 132 133 134 135 136 138 139 143 144 145 146
 147 151 155 156 157 158 159 160 162 163 167 168]


In [10]:
# Process in batches if plot is needed to avoid memory issues (restart kernel between batches)
process_batch = False
if process_batch:
    i = 0 
    batch_size = 5
    sel_sequence_id = sel_sequence_id[i*batch_size:(i+1)*batch_size]
    plot = True
    savefig = True

df_processed = build_arrivals_dataset(
    df=df_sel,
    ds_gps=ds_gps,
    sel_sequence_id=sel_sequence_id,
    pre_reception_time=pre_reception_time,
    post_reception_time=post_reception_time,
    img_root=img_process_folder,
    channels_order=channels_order,
    used_channel=used_channel,
    plot=plot,
    plot_zoom=False,
    savefig=savefig,
    verbose=False,
)

if not process_batch:
    ds_processed = xr.Dataset.from_dataframe(df_processed)
    fpath_save = os.path.join(
        data_folder,
        f"processed_arrivals.nc",
    )
    ds_processed.to_netcdf(fpath_save)

Selected sequences: 84 -> [  1   2  12  13  14  15  16  24  25  26  27  28  29  35  39  40  41  42
  43  44  46  47  51  55  59  60  61  62  63  64  69  70  71  72  76  80
  81  82  83  84  85  90  91  95  99 100 101 102 103 104 106 110 116 117
 118 119 120 121 123 127 131 132 133 134 135 136 138 139 143 144 145 146
 147 151 155 156 157 158 159 160 162 163 167 168]
Wrong number of arrivals detected
Wrong number of arrivals detected


ValueError: All arrays must be of the same length

In [ ]:
df_processed

In [ ]:
# for sel_id in df_processed["Sequence_id"].unique():
#     df_seq = df_processed[df_processed["Sequence_id"] == sel_id]

#     for obs_id in [1, 2, 3]:

#         # First criterion: ratio of detected arrivals
#         col_name = f"Valid detection OBS{obs_id}"
#         n_detected = df_seq[col_name].sum()
#         crit_1 = n_detected / df_seq.shape[0]

#         # Second criterion: error relative to expected repetition period
#         col_name = f"Arrival datetime OBS{obs_id}"
#         t_diff_mean = df_seq[col_name].diff().mean().total_seconds()
#         repeat_period_em = df_seq["Trepeat (s)"].iloc[0]
#         crit_2 = 1 - abs(t_diff_mean - repeat_period_em) / repeat_period_em
#         if t_diff_mean < 0 or crit_2 < 0 or np.isnan(crit_2):
#             crit_2 = 0

#         # Third criterion: normalized psnr
#         col_name = f"PSNR OBS{obs_id}"
#         psnr_mean = df_seq[col_name].mean()
#         crit_3 = psnr_mean / df_processed[col_name].max()
#         if np.isnan(crit_3):
#             crit_3 = 0

#         # Final score as sum of criteria
#         final_score = (crit_1 + crit_2 + crit_3) / 3
#         print(
#             f"Sequence ID {sel_id} - OBS{obs_id} : Score = {final_score:.2f} (C1={crit_1:.2f}, C2={crit_2:.2f}, C3={crit_3:.2f})"
#         )

#         # Store final score in dataframe
#         df_processed.loc[
#             (df_processed["Sequence_id"] == sel_id),
#             f"f_score OBS{obs_id}",
#         ] = final_score




In [ ]:
# df_processed_valid = df_processed.loc[
#     df_processed["Valid detection OBS1"]
#     & df_processed["Valid detection OBS2"]
#     & df_processed["Valid detection OBS3"]
# ]